<a href="https://colab.research.google.com/github/Rumas0/Thesis_work_SSL-imbalance/blob/main/Real_Data_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Mounting Drive and importing Libraries**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu


**Loading Data**

In [8]:
backup_dir = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'

train_df = pd.read_csv(f'{backup_dir}/expA_train.csv')
val_df = pd.read_csv(f'{backup_dir}/expA_val.csv')
test_df = pd.read_csv(f'{backup_dir}/expA_test.csv')

IMAGE_DIR = 'data/labeled_real'

class SkinDataset(Dataset):
  def __init__(self, df, transform=None):
    self.df = df
    self.transform = transform
    self.classes = sorted(df['label'].unique())
    self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    img = Image.open(f"{IMAGE_DIR}/{row['image']}.jpg").convert('RGB')
    label = self.class_to_idx[row['label']]
    if self.transform:
      img = self.transform(img)
    return img, label


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.455, 0.406], [0.229, 0.224, 0.225])
])

train_ds = SkinDataset(train_df, transform)
val_ds = SkinDataset(val_df, transform)
test_ds = SkinDataset(test_df, transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"Classes: {train_ds.classes}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"Train Distribution:", train_df['label'].value_counts().to_dict())

Classes: ['BKL', 'MEL', 'NV']
Train: 483, Val: 104, Test: 104
Train Distribution: {'NV': 349, 'BKL': 99, 'MEL': 35}


**FOCAL LOSS-- better than class weights previously used for synthetic data**

In [11]:
class FocalLoss(nn.Module):
  def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
    super().__init__()
    self.alpha = alpha ##Weighting Class
    self.gamma = gamma
    self.reduction = reduction

  def froward(self, inputs, targets):
    ce_loss = F.cross_entropy(input, targets, reduction='noone', weight=self.alpha)
    pt = torch.exp(-ce_loss)
    focal_term = (1 - pt) ** self.gamma
    loss = focal_term * ce_loss

    if self.reduction == 'mean':
      return loss.mean()
    elif self.reduction == 'sum':
      return loss.sum()
    return loss

##**Model Architecture**

In [12]:
class SimpleEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = nn.Sequectial(
        nn.conv2d(3, 32, 3, paddings=1), nn.Relu(), nn.MaxPool2d(2),
        nn.conv2d(32, 64, 3, padding=1), nn.Relu(), nn.MaxPool2d(2),
        nn.conv2d(64, 128, 3, padding=1), nn.Relu(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten()
    )

  def forward(self, x):
    return self.features(x)

class SSLClassifier(nn.Module):
  def __init__(self, encoder, num_classes):
    super().__init__()
    self.encoder = encoder
    self.classifier = nn.Sequential(
        nn.Linear(128, 64), nn.Relu(), nn.Dropout(0.3),
        nn.Linear(64, num_classes)
    )

  def forward(self, x):
    return self.classifier(self.encoder(x))